# SSU Data Dash: quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/erwinzhang7/SSU-Data-Dash/blob/main/notebooks/quickstart.ipynb)

The same steps as `python run.py validate / report / predict / check`, one cell at a time.
Run the cells from top to bottom. Colab already has every package this needs.

In [ ]:
# Setup. In Colab this downloads the starter code; on your own computer it just
# moves to the repo folder.
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/SSU-Data-Dash"):
        !git clone -q https://github.com/erwinzhang7/SSU-Data-Dash.git /content/SSU-Data-Dash
    os.chdir("/content/SSU-Data-Dash")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

# Reload features.py and models.py automatically after you edit them.
%load_ext autoreload
%autoreload 2

## 1. Get the data

The files download from Hugging Face the first time and are cached for later runs. If you downloaded them by hand, set `DATA_DIR` in the next cell instead.

In [ ]:
# os.environ["DATA_DIR"] = "/path/to/folder"   # only if you downloaded the files yourself

import data
listings = data.load("listings")
print(f"{len(listings):,} sales, {listings['date_sold'].min():%B %Y} to {listings['date_sold'].max():%B %Y}")
listings.head()

In [ ]:
off_market = data.load("off_market")
list_aware = data.load("list_aware")
print(f"{len(off_market):,} homes to value with no list price (off-market)")
print(f"{len(list_aware):,} homes to value that have a list price (list-aware)")
off_market.head()

In [ ]:
# A first look: the median sold price each month, by property type.
monthly = listings.groupby([listings["date_sold"].dt.to_period("M"), "house_type_name"])["sold_price"].median()
monthly.unstack().plot(figsize=(9, 4), ylabel="Median sold price ($)", xlabel="");

## 2. Validate

Each month from December 2025 to May 2026 is predicted by models trained only on the sales
before it, the same way the real test works. This takes about a minute.

In [ ]:
import run
run.validate()

In [ ]:
run.report()
from IPython.display import Image
Image("figures/error_by_price_range.png")

## 3. Improve the models

Edit **features.py** and **models.py**, then rerun the two cells above to see whether your
validation error goes down. In Colab, open them from the folder icon on the left
(`SSU-Data-Dash`), edit, and save with Ctrl+S (Cmd+S on a Mac).
Colab deletes your files when the session ends, so download anything you change.

## 4. Predict and check

This fits the models on all 150,000 sales and writes the two submission files.

In [ ]:
run.predict()
run.check()

In [ ]:
# In Colab: download the two submission files.
if "google.colab" in sys.modules:
    from google.colab import files
    files.download("submissions/submission_off_market.csv")
    files.download("submissions/submission_list_aware.csv")